# Kapitel 3: Wörter werden zu Zahlen

## Bisher...

Wir haben aus 3,6 Millionen echten Amazon-Bewertungen eine Stichprobe von ~400.000 gezogen (Kap. 1),
die natürliche Verteilung analysiert, das Klassenungleichgewicht durch Undersampling korrigiert
und die Texte normalisiert (Kap. 2).

## Das Problem

Eine Maschine kann nicht lesen. Sie versteht weder *"great"* noch *"terrible"*.
Wir müssen jeden Text in einen **numerischen Vektor** verwandeln —
so, dass die Bedeutung erhalten bleibt.

## Die Lösung: TF-IDF

**TF-IDF** (Term Frequency – Inverse Document Frequency) misst die Relevanz eines Wortes:
- **TF** — Wie oft kommt das Wort in *diesem* Review vor?
- **IDF** — Wie selten ist es im *gesamten* Datensatz?
- **TF × IDF** — Einzigartig-wichtige Wörter erhalten hohe Werte

## 3.1 Bereinigte Daten laden

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviews – Tokenisierung & TF-IDF") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

df = spark.read.parquet(
    "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/cleaned_reviews.parquet"
)

print(f"Daten geladen: {df.count():,} Zeilen")
df.printSchema()
df.show(3, truncate=80)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/28 22:49:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/28 22:49:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/28 22:49:05 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


## 3.2 Schritt 1: Tokenisierung

*"this book is great"* → `["this", "book", "is", "great"]`

In [ ]:
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(inputCol="text", outputCol="tokens")
df_tokens = tokenizer.transform(df)
df_tokens.select("text", "tokens").show(3, truncate=80)

In [ ]:
from pyspark.sql.functions import size, avg, min as spark_min, max as spark_max, round as spark_round, col

df_tokens.withColumn("token_count", size("tokens")).select(
    spark_min("token_count").alias("Min"),
    spark_round(avg("token_count"), 0).alias("Durchschnitt"),
    spark_max("token_count").alias("Max")
).show()

## 3.3 Schritt 2: StopWords entfernen

`["this", "book", "is", "great"]` → `["book", "great"]`

In [ ]:
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(inputCol="tokens", outputCol="filtered_tokens")
df_filtered = remover.transform(df_tokens)
df_filtered.select("tokens", "filtered_tokens").show(3, truncate=80)

In [ ]:
df_compare = df_filtered \
    .withColumn("before", size("tokens")) \
    .withColumn("after", size("filtered_tokens")) \
    .withColumn("removed", col("before") - col("after"))

df_compare.select(
    spark_round(avg("before"), 1).alias("Tokens vorher"),
    spark_round(avg("after"), 1).alias("Tokens nachher"),
    spark_round(avg("removed"), 1).alias("Entfernt (Durchschnitt)")
).show()

## 3.4 Welche Wörter verraten die Stimmung?

In [ ]:
from pyspark.sql.functions import explode, count as spark_count

df_words = df_filtered.select("label", explode("filtered_tokens").alias("word"))

print("=== Top 15 Wörter – POSITIV ===")
df_words.filter(col("label") == 1) \
    .groupBy("word").agg(spark_count("*").alias("count")) \
    .orderBy(col("count").desc()).show(15)

print("=== Top 15 Wörter – NEGATIV ===")
df_words.filter(col("label") == 0) \
    .groupBy("word").agg(spark_count("*").alias("count")) \
    .orderBy(col("count").desc()).show(15)

## 3.5 Schritt 3: TF-IDF berechnen

In [ ]:
from pyspark.ml.feature import HashingTF, IDF

hashing_tf = HashingTF(inputCol="filtered_tokens", outputCol="raw_features", numFeatures=10000)
df_tf = hashing_tf.transform(df_filtered)
print("HashingTF abgeschlossen.")
df_tf.select("filtered_tokens", "raw_features").show(2, truncate=80)

In [ ]:
idf = IDF(inputCol="raw_features", outputCol="tfidf_features")
idf_model = idf.fit(df_tf)
df_tfidf = idf_model.transform(df_tf)
print("IDF-Berechnung abgeschlossen.")
df_tfidf.select("label", "filtered_tokens", "tfidf_features").show(2, truncate=80)

## 3.6 Feature-Vektor inspizieren

In [ ]:
sample = df_tfidf.select("tfidf_features").first()[0]

print(f"Vektor-Typ:   {type(sample).__name__}")
print(f"Dimensionen:  {sample.size}")
print(f"Nicht-Null:   {len(sample.indices)} Einträge")

## 3.7 Ergebnisse speichern

In [ ]:
df_output = df_tfidf.select("label", "text", "filtered_tokens", "tfidf_features")

output_path = "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/tfidf_features.parquet"
df_output.write.parquet(output_path, mode="overwrite")
print(f"Gespeichert: {output_path}")

df_check = spark.read.parquet(output_path)
print(f"Parquet gelesen: {df_check.count():,} Zeilen")

## 3.8 Visualisierung

In [ ]:
import matplotlib.pyplot as plt

top_pos = df_words.filter(col("label") == 1) \
    .groupBy("word").agg(spark_count("*").alias("count")) \
    .orderBy(col("count").desc()).limit(15).toPandas()

top_neg = df_words.filter(col("label") == 0) \
    .groupBy("word").agg(spark_count("*").alias("count")) \
    .orderBy(col("count").desc()).limit(15).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(top_pos["word"][::-1], top_pos["count"][::-1], color="#5DCAA5", edgecolor="#0F6E56")
axes[0].set_title("Top 15 Wörter – POSITIV")
axes[0].set_xlabel("Häufigkeit")
axes[1].barh(top_neg["word"][::-1], top_neg["count"][::-1], color="#F09595", edgecolor="#A32D2D")
axes[1].set_title("Top 15 Wörter – NEGATIV")
axes[1].set_xlabel("Häufigkeit")
plt.tight_layout()
plt.show()

## Kapitel 3 — Zusammenfassung

| Schritt | Werkzeug | Was passiert? |
|---------|----------|---------------|
| Tokenisierung | `Tokenizer` | Text → Wortliste |
| StopWords | `StopWordsRemover` | Bedeutungslose Wörter entfernen |
| TF | `HashingTF(10000)` | Wörter → Häufigkeits-Vektoren |
| IDF | `IDF` | Häufigkeiten → Relevanz-Gewichtung |

**Nächstes Kapitel:** Die Maschine lernt — Logistic Regression auf echten Trainingsdaten.